In [ ]:
from langchain.chat_models import init_chat_model
from open_deep_research.knowledge import (
    dataset_info,
    abbreviation
)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_experimental.utilities import PythonREPL
import textwrap
import os
# os.load_dotenv()

from open_deep_research.prompts import (
    report_planner_query_writer_instructions,
    report_planner_instructions,
    query_writer_instructions,
    section_writer_instructions_v2,
    section_writer_instructions,
    final_section_writer_instructions,
    final_section_writer_instructions_v2,
    section_grader_instructions,
    section_writer_inputs,
    visualization_instructions,
    sql_instructions,
    sql_grader_instructions,
    visualization_instructions
)
from typing import Annotated, List, TypedDict, Literal
from pydantic import BaseModel, Field
import operator

sql_interpret_instruction = """You are **Data Insight Agent**, an analytical assistant that turns raw SQL outputs into concise, executive-ready briefings.
<sql_result>
{sql_result}
</sql_result>"""

class SQLResponse(BaseModel):
    sql_script: str = Field(None, description="SQL script if cant retrive output empty string ")
    explaination: str = Field(None, description="explain why cant retrive given data schema, else ouput empty")


class ClarifyQuestion(BaseModel):
    follow_up_questions: List[str] = Field(None, description="Alternative questions to query if there is not enough information in original question")

class SQLInterpret(BaseModel):
    insight: str = Field(None, description="Insight about sql result and question")
    summarization:  str = Field(None, description="Summarization about sql result and question")

class VisualizeScript(BaseModel):
    visualize_script: str = Field(None, description="visualize script if cant visulize output empty string ")


llm_flash = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0,
)

# llm_flash = llm_flash.with_structured_output(SQLResponse)
llm_4o = ChatOpenAI(
    model_name="gpt-4o-mini",      # or "gpt-4o-mini" for lower cost / latency
    
    temperature=0.0,
    # Optional:
    # max_tokens=2048,
    # timeout=30,              # seconds
    # base_url="https://api.openai.com/v1",
)
import textwrap
from google.cloud import bigquery

def python_execute(code: str) -> str:
    """
    Execute Python code in a fresh PythonREPL and return stdout / errors.
    """
    import traceback
    import sys

    wrapped = "try:\n"
    wrapped += textwrap.indent(code, "    ")
    wrapped += textwrap.dedent(
        """
        except Exception as e:
            print(f"error: {e}")
    """
    )
    # print(wrapped)
    python_repl = PythonREPL()
    return python_repl.run(wrapped)

def query_bigquery(sql: str, project_id: str = "agentic-ai-463517") -> str:
    """
    Run a BigQuery query in a throw-away PythonREPL sandbox and return its stdout.
    Using `repr(sql)` guarantees the SQL is embedded as a *single* clean string.
    """
    code = f"""
from google.cloud import bigquery
client = bigquery.Client(project={project_id!r})
query_job = client.query({sql!r})
if query_job.result().total_rows == 0:
    print("No results found.")
else:
    print(list(query_job.result()))
"""
    return python_execute(code)


# Wrap with the Pydantic schema so every call returns a parsed SQLResponse.
# llm_4o = llm_4o.with_structured_output(SQLResponse)
def sql_writer(llm, question):
    print("sql writer.....")
    project_id = "agentic-ai-463517"
    dataset_name = "ghn_data"
    sql_llm = llm.with_structured_output(SQLResponse)
    clarify_sql_llm = llm.with_structured_output(ClarifyQuestion)
    system_instructions_query = sql_instructions.format(
                        question=question,
                        dataset_info=dataset_info,
                        project_id=project_id,
                        dataset_name=dataset_name,
                        last_error="there was no error in the last query",
                        previous_query="",
                    )
        # print(system_instructions_query)
    result =  sql_llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])
    print("="*100)
    #print(system_instructions_query)
    #print(result.sql_script)
    #print(result.explaination)
    print("="*100)
    
    if result.sql_script != "":
        output = query_bigquery(result.sql_script)
        print(output)
        print("="*100)
        for i in range(3):
            print(result)
            print(output)
            if "error" in output:
                system_instructions_query = sql_instructions.format(
                                question=question,
                                dataset_info=dataset_info,
                                project_id=project_id,
                                dataset_name=dataset_name,
                                last_error=output,
                                previous_query=result.sql_script,
                            )
                print(system_instructions_query)
                result =  sql_llm.invoke([SystemMessage(content=system_instructions_query),
                                                                        HumanMessage(content=question)])
                output = query_bigquery(result.sql_script)
                print("="*100)
                #print(result.sql_script)
                #print(result.explaination)
                print(output)
                print("="*100)
            else:
                return output
    else:
        # adding follow up question
        print('Cant write sql')
        print(result.explaination)
        result =  clarify_sql_llm.invoke([SystemMessage(content=system_instructions_query),
                                                                        HumanMessage(content=question)])
        print(result.follow_up_questions)

def visualize_data(llm, question, sql_result):
    llm = llm.with_structured_output(VisualizeScript)

    system_instructions_query = visualization_instructions.format(
                        previous_script="",
                        query_result=sql_result,
                        folder_path="./",
                        last_error="",
                    )
        # print(system_instructions_query)
    result =  llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])

    print(result.visualize_script)
    python_execute(result.visualize_script)

def reasoning_data(llm, question, sql_result):
    llm = llm.with_structured_output(SQLInterpret)

    system_instructions_query = sql_interpret_instruction.format(
                        sql_result=sql_result
                    )
        # print(system_instructions_query)
    result =  llm.invoke([SystemMessage(content=system_instructions_query),
                                                            HumanMessage(content=question)])
    print(result.summarization)
    print(result.insight)




In [18]:
sql = sql_writer(llm_4o,"What is the main goal of improving NVPTTT productivity by 15% and its expected financial impact?")


sql writer.....
Cant write sql
The question does not relate to the provided data schema, which includes tables for shipping orders, locations, warehouses, middle mile logs, revenue orders, and SLA delivery. There are no columns or tables that pertain to NVPTTT productivity or its financial impact.
['What specific metrics or KPIs are used to measure NVPTTT productivity?', 'What is the current financial impact of NVPTTT productivity?', 'Are there specific financial figures or projections available for the expected impact of the 15% productivity improvement?', 'What timeframe is considered for achieving the 15% productivity improvement?']
